# Анализ пролонгаций договоров за 2023 год

**Логика расчёта коэффициентов:**

Для каждого целевого месяца T:
- **Коэфф_1** = отгрузка за T проектов, завершившихся в T-1 и пролонгированных в T / отгрузка за T-1 всех проектов, завершившихся в T-1
- **Коэфф_2** = отгрузка за T проектов, завершившихся в T-2, не пролонгированных в T-1, но пролонгированных в T / отгрузка за T-2 проектов, завершившихся в T-2 и не пролонгированных в T-1

# Импорты и константы

In [91]:
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta
from pathlib import Path
from typing import Iterator
from string import Template

In [92]:
BASE_DIR = Path().resolve()

PROLONGATIONS_PATH  = BASE_DIR / 'data' / 'prolongations.csv'
FINANCIAL_DATA_PATH = BASE_DIR / 'data' / 'financial_data.csv'
OUTPUT_REPORT_TEMPLATE = Template('Prolongation_Report_${year}.xlsx')
CHUNK_SIZE  = 10_000
TARGET_YEAR = 2023

RU_MONTHS = {
    'январь': '01', 'февраль': '02', 'март':    '03', 'апрель':  '04',
    'май':    '05', 'июнь':    '06', 'июль':     '07', 'август':  '08',
    'сентябрь': '09', 'октябрь': '10', 'ноябрь': '11', 'декабрь': '12'
}


def to_yyyymm(ru_str: str) -> str:
    """
    Превращает "Ноябрь 2022" → "2022-11".
    Если формат не распознан — возвращает исходную строку без изменений.
    """
    parts = str(ru_str).strip().lower().split()
    if len(parts) == 2:
        month_num = RU_MONTHS.get(parts[0])
        if month_num:
            return f"{parts[1]}-{month_num}"
    return str(ru_str)

# Формирование rename_map из заголовков financial_data.csv

Читаем только первую строку, чтобы получить имена колонок без загрузки всего файла.
Строим словарь `{«Ноябрь 2022»: «2022-11», ...}` для всех месячных колонок.

In [93]:
# Читаем только заголовок - nrows=0 не грузит данные
header_df = pd.read_csv(FINANCIAL_DATA_PATH, nrows=0)

# Служебные колонки, которые НЕ являются месяцами
NON_MONTH_COLS = {'id', 'Причина дубля', 'Account'}

# rename_map: "Ноябрь 2022" -> "2022-11" только для реальных месячных колонок
rename_map = {
    col: to_yyyymm(col)
    for col in header_df.columns
    if col not in NON_MONTH_COLS
}

# Отсортированный список месяцев в формате YYYY-MM
sorted_months_yyyymm = sorted(rename_map.values())

print(f"Найдено месячных колонок: {len(rename_map)}")
print(f"Диапазон: {sorted_months_yyyymm[0]} — {sorted_months_yyyymm[-1]}")

Найдено месячных колонок: 16
Диапазон: 2022-11 — 2024-02


# Генератор чтения CSV чанками

In [94]:
def read_csv_gen(filepath: str | Path, chunk_size: int = CHUNK_SIZE) -> Iterator[pd.DataFrame]:
    """Читает CSV файл порциями (чанками), не загружая всё в память сразу."""
    print(f"Чтение {filepath} чанками по {chunk_size} строк...")
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        yield chunk

# Генератор фильтрации и создания признаков

Для каждого чанка:
1. Удаляем служебные колонки (`Причина дубля`, `Account`)
2. Переименовываем месяцы в формат `YYYY-MM`
3. Создаём признаки: `_stop` (флаг стоп/end), `_vnol` (флаг «в ноль»), `_val` (числовое значение)
4. Делаем локальную агрегацию по `id` внутри чанка

In [95]:
def filter_and_process_gen(
    raw_chunks_gen: Iterator[pd.DataFrame],
    rename_map: dict,
) -> Iterator[pd.DataFrame]:
    """
    Обрабатывает чанки: создаёт признаки для каждого месяца и агрегирует по id.
    """
    month_cols_yyyymm = sorted(rename_map.values())

    for chunk in raw_chunks_gen:
        # Удаляем колонки, не нужные для расчёта
        # errors='ignore' - если колонки нет, не падаем
        chunk = chunk.drop(columns=['Причина дубля', 'Account'], errors='ignore')
        chunk = chunk.rename(columns=rename_map)

        agg_dict = {}
        for m in month_cols_yyyymm:
            if m not in chunk.columns:
                continue

            col_str = chunk[m].astype(str).str.strip().str.lower()

            # Флаг "стоп": проект завершился досрочно
            chunk[f'{m}_stop'] = col_str.isin(['стоп', 'end'])
            # Флаг "в ноль": отгрузка нулевая, нужно взять значение предыдущего месяца
            chunk[f'{m}_vnol'] = (col_str == 'в ноль')
            # Числовое значение отгрузки; нечисловые строки -> 0.0
            # Приводим денежный формат к разделителю точке, чтобы корректно парсить в число
            clean_val = chunk[m].astype(str).str.replace(r'\s+', '', regex=True).str.replace(',', '.') 
            chunk[f'{m}_val']  = pd.to_numeric(clean_val, errors='coerce').fillna(0.0)

            agg_dict[f'{m}_stop'] = 'max'
            agg_dict[f'{m}_vnol'] = 'max'
            agg_dict[f'{m}_val']  = 'sum'

        # Исходные месячные колонки больше не нужны
        chunk = chunk.drop(columns=month_cols_yyyymm, errors='ignore')

        # Локальная агрегация: схлопываем дубли id внутри чанка
        processed_chunk = chunk.groupby('id').aggregate(agg_dict).reset_index()
        yield processed_chunk

# Сборка основного датасета

1. Объединяем все обработанные чанки в один DataFrame
2. Применяем логику «в ноль»: если в месяц T стоит флаг `_vnol` и числовая сумма = 0, переносим значение из T-1
3. Применяем логику «стоп»: исключаем проекты, у которых есть `_stop` в последний месяц или ранее
4. Вычисляем `Shipment_Base`, `Shipment_M1`, `Shipment_M2` для расчёта коэффициентов

In [ ]:
def build_core_dataset(
    prol_path: str | Path,
    processed_chunks_gen,
    sorted_months_yyyymm: list
) -> pd.DataFrame:
    """
    Собирает финальный датасет:
    - Объединяет чанки и убирает дубли по id
    - Мерджит со справочником пролонгаций
    - Применяет бизнес-логику "в ноль" и "стоп"
    - Добавляет колонки Shipment_Base / M1 / M2
    """
    print("Агрегация обработанных чанков и применение бизнес-логики...")

    # Собираем все чанки и делаем финальную агрегацию по id.
    # Для _stop/_vnol используем max (True побеждает), для _val — sum.
    all_chunks = pd.concat(processed_chunks_gen)

    val_cols  = [c for c in all_chunks.columns if c.endswith('_val')]
    flag_cols = [c for c in all_chunks.columns if c.endswith('_stop') or c.endswith('_vnol')]

    agg_final = {c: 'sum' for c in val_cols}
    agg_final.update({c: 'max' for c in flag_cols})

    df_fin = all_chunks.groupby('id').aggregate(agg_final).reset_index()

    # Справочник пролонгаций.
    # AM из справочника первичен относительно Account из financial_data.
    df_prol = pd.read_csv(prol_path).rename(columns={'month': 'end_month', 'AM': 'Manager'})

    # Конвертируем end_month из "Ноябрь 2022" в "2022-11" если нужно
    df_prol['end_month'] = df_prol['end_month'].apply(
        lambda x: to_yyyymm(x) if not str(x).count('-') == 1 else x
    )

    # берём только проекты, присутствующие в обоих файлах
    df = df_prol.merge(df_fin, on='id', how='inner')

    # Логика "в ноль"
    # Если в текущем месяце T стоит флаг _vnol И числовая сумма = 0,
    # значит все части оплаты равны нулю, то берём значение предыдущего месяца T-1.
    for i in range(1, len(sorted_months_yyyymm)):
        curr_m, prev_m = sorted_months_yyyymm[i], sorted_months_yyyymm[i - 1]

        if f'{curr_m}_vnol' not in df.columns:
            continue

        mask_vnol = df[f'{curr_m}_vnol'] & (df[f'{curr_m}_val'] == 0)
        df.loc[mask_vnol, f'{curr_m}_val'] = df.loc[mask_vnol, f'{prev_m}_val']

    # Логика "стоп" 
    # Исключаем проект, если в его последний месяц (end_month) или раньше
    # встречается флаг _stop == True.
    # Условие: end_dt >= m_dt означает «месяц m не позже конца проекта»,
    # т.е. стоп произошёл в период реализации.
    df['is_excluded'] = False
    df['end_dt'] = pd.to_datetime(df['end_month'], errors='coerce') 

    # Строки с непарсируемым end_month сразу исключаем - данные некорректны
    df.loc[df['end_dt'].isna(), 'is_excluded'] = True

    for m in sorted_months_yyyymm:
        if f'{m}_stop' not in df.columns:
            continue
        m_dt = pd.to_datetime(m)
        # Стоп засчитывается только если он случился в период реализации проекта
        mask_stop = (df['end_dt'] >= m_dt) & df[f'{m}_stop']
        df.loc[mask_stop, 'is_excluded'] = True

    df_clean = df[~df['is_excluded']].copy()

    # ── Расчёт базовых отгрузок ───────────────────────────────────────────────
    # Shipment_Base - отгрузка в последний месяц реализации (end_month)
    # Shipment_M1 - отгрузка в следующий месяц после окончания
    # Shipment_M2 - отгрузка через два месяца после окончания
    def get_shifted_val(row, shift):
        if pd.isna(row['end_dt']):
            return 0.0
        target_col = (row['end_dt'] + relativedelta(months=shift)).strftime('%Y-%m') + '_val'
        return row[target_col] if target_col in row.index else 0.0

    df_clean['Shipment_Base'] = df_clean.apply(lambda r: get_shifted_val(r, 0), axis=1)
    df_clean['Shipment_M1'] = df_clean.apply(lambda r: get_shifted_val(r, 1), axis=1)
    df_clean['Shipment_M2'] = df_clean.apply(lambda r: get_shifted_val(r, 2), axis=1)

    print(f"Итого проектов в core dataset: {len(df_clean)}")
    return df_clean

# Расчёт коэффициентов пролонгации

Для каждого месяца T года:

**Коэфф_1** (пролонгация в первый месяц):
- Числитель: Σ Shipment_M1 проектов, завершившихся в T-1, у которых Shipment_M1 > 0
- Знаменатель: Σ Shipment_Base всех проектов, завершившихся в T-1

**Коэфф_2** (пролонгация во второй месяц):
- База: проекты, завершившиеся в T-2 и НЕ пролонгированные в T-1 (Shipment_M1 == 0)
- Числитель: Σ Shipment_M2 проектов из базы, у которых Shipment_M2 > 0
- Знаменатель: Σ Shipment_Base проектов из базы

In [97]:
def calc_statistics_gen(core_df: pd.DataFrame, target_year: int):
    """
    Генератор: для каждого месяца target_year возвращает DataFrame
    с коэффициентами пролонгации по каждому менеджеру.
    """
    print(f"Расчёт коэффициентов пролонгации за {target_year} год...")
    target_months = [f"{target_year}-{str(i).zfill(2)}" for i in range(1, 13)]
    all_managers = core_df[['Manager']].drop_duplicates()

    for target_month in target_months:
        t_dt = pd.to_datetime(target_month)
        month_minus_1 = (t_dt - relativedelta(months=1)).strftime('%Y-%m')
        month_minus_2 = (t_dt - relativedelta(months=2)).strftime('%Y-%m')

        # Коэффициент 1
        # Проекты, завершившиеся в прошлом месяце
        df_c1 = core_df[core_df['end_month'] == month_minus_1].copy()

        # Знаменатель: вся отгрузка завершившихся проектов за последний месяц
        c1_denom = (
            df_c1.groupby('Manager')['Shipment_Base'].sum().reset_index(name='C1_Denom')
        )

        # Числитель: отгрузка в T только тех проектов, которые реально пролонгировались
        df_c1['C1_Num_Val'] = np.where(df_c1['Shipment_M1'] > 0, df_c1['Shipment_M1'], 0)
        c1_num = (
            df_c1.groupby('Manager')['C1_Num_Val'].sum().reset_index(name='C1_Num')
        )

        # Коэффициент 2 
        # Проекты, завершившиеся два месяца назад И не пролонгированные в прошлом месяце
        df_c2 = core_df[
            (core_df['end_month'] == month_minus_2) &
            (core_df['Shipment_M1'] == 0) # не было отгрузки в T-1
        ].copy()

        # Знаменатель: отгрузка этих проектов за их последний месяц
        c2_denom = (
            df_c2.groupby('Manager')['Shipment_Base'].sum().reset_index(name='C2_Denom'))

        # Числитель: из них те, что пролонгировались в T
        df_c2['C2_Num_Val'] = np.where(df_c2['Shipment_M2'] > 0, df_c2['Shipment_M2'], 0)
        c2_num = (
            df_c2.groupby('Manager')['C2_Num_Val'].sum().reset_index(name='C2_Num')
        )

        # Сборка результата за месяц 
        month_res = all_managers.copy()
        month_res['Target_Month'] = target_month

        for part in [c1_denom, c1_num, c2_denom, c2_num]:
            month_res = month_res.merge(part, on='Manager', how='left')

        month_res.fillna(0, inplace=True)

        # Коэффициенты: защита от деления на ноль
        month_res['Коэфф_1_Месяц'] = np.where(
            month_res['C1_Denom'] == 0, 0,
            month_res['C1_Num'] / month_res['C1_Denom']
        )
        month_res['Коэфф_2_Месяц'] = np.where(
            month_res['C2_Denom'] == 0, 0,
            month_res['C2_Num'] / month_res['C2_Denom']
        )

        yield month_res

# Расчёт годовых коэффициентов

Годовой коэффициент считается как отношение суммарного числителя к суммарному знаменателю за все 12 месяцев.

In [98]:
def calc_annual_stats(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """
    Вычисляет годовые коэффициенты как отношение суммарных числителей к знаменателям.
    """
    annual = (
        monthly_df
        .groupby('Manager')[['C1_Num', 'C1_Denom', 'C2_Num', 'C2_Denom']]
        .sum()
        .reset_index()
    )

    annual['Коэфф_1_Год'] = np.where(
        annual['C1_Denom'] == 0, 0,
        annual['C1_Num'] / annual['C1_Denom']
    )
    annual['Коэфф_2_Год'] = np.where(
        annual['C2_Denom'] == 0, 0,
        annual['C2_Num'] / annual['C2_Denom']
    )

    return annual

# Формирование и сохранение Excel-отчёта

Отчёт состоит из трёх листов:
1. **Сводная** — годовые коэффициенты по каждому менеджеру + строка «Весь отдел»
2. **По месяцам** — помесячная разбивка коэффициентов для каждого менеджера
3. **Данные** — исходные числители и знаменатели для проверки

In [99]:
def build_and_save_report(
    monthly_df: pd.DataFrame,
    annual_df:  pd.DataFrame,
    target_year: int,
    output_dir:  Path = BASE_DIR
) -> Path:
    output_path = output_dir / OUTPUT_REPORT_TEMPLATE.substitute(year=target_year)

    # Строка «Весь отдел»
    dept = annual_df[['C1_Num', 'C1_Denom', 'C2_Num', 'C2_Denom']].sum().to_frame().T
    dept['Manager'] = 'Весь отдел'
    dept['Коэфф_1_Год']  = np.where(dept['C1_Denom'] == 0, 0, dept['C1_Num'] / dept['C1_Denom'])
    dept['Коэфф_2_Год']  = np.where(dept['C2_Denom'] == 0, 0, dept['C2_Num'] / dept['C2_Denom'])
    summary_df = pd.concat([annual_df, dept], ignore_index=True)

    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        summary_df.rename(columns={
            'Manager':      'Менеджер',
            'Коэфф_1_Год':  'Коэфф. 1 (год)',
            'Коэфф_2_Год':  'Коэфф. 2 (год)',
            'C1_Num':       'Числитель К1',
            'C1_Denom':     'Знаменатель К1',
            'C2_Num':       'Числитель К2',
            'C2_Denom':     'Знаменатель К2',
        }).to_excel(writer, sheet_name='Сводная', index=False)

        monthly_df.rename(columns={
            'Manager':        'Менеджер',
            'Target_Month':   'Месяц',
            'Коэфф_1_Месяц':  'Коэфф. 1',
            'Коэфф_2_Месяц':  'Коэфф. 2',
        })[['Менеджер', 'Месяц', 'Коэфф. 1', 'Коэфф. 2']].to_excel(
            writer, sheet_name='По месяцам', index=False
        )

    print(f"Отчёт сохранён: {output_path}")
    return output_path

# Запуск пайплайна

In [100]:
# Цепочка генераторов
raw_gen = read_csv_gen(FINANCIAL_DATA_PATH, CHUNK_SIZE)
processed_gen = filter_and_process_gen(raw_gen, rename_map)

# Собираем основной датасет
core_df = build_core_dataset(
    prol_path            = PROLONGATIONS_PATH,
    processed_chunks_gen = processed_gen,
    sorted_months_yyyymm = sorted_months_yyyymm
)

# Считаем помесячную статистику
monthly_results = calc_statistics_gen(core_df, TARGET_YEAR)
monthly_df = pd.concat(monthly_results, ignore_index=True)

# Считаем годовые коэффициенты
annual_df = calc_annual_stats(monthly_df)

print("\n-- Годовые коэффициенты пролонгации --")
print(annual_df[['Manager', 'Коэфф_1_Год', 'Коэфф_2_Год']].to_string(index=False))

# Сохраняем отчёт в Excel
report_path = build_and_save_report(monthly_df, annual_df, TARGET_YEAR)

Агрегация обработанных чанков и применение бизнес-логики...
Чтение C:\Users\user\Desktop\ТЗ-Data-Analysis\data\financial_data.csv чанками по 10000 строк...
Итого проектов в core dataset: 429
Расчёт коэффициентов пролонгации за 2023 год...

-- Годовые коэффициенты пролонгации --
                      Manager  Коэфф_1_Год  Коэфф_2_Год
 Васильев Артем Александрович     0.523233     0.062796
      Иванова Мария Сергеевна     0.351146     0.000000
     Кузнецов Михаил Иванович     0.478448     0.000000
    Михайлов Андрей Сергеевич     0.664941     0.000000
      Петрова Анна Дмитриевна     1.111182     0.000000
  Попова Екатерина Николаевна     0.511768     0.033018
  Смирнова Ольга Владимировна     0.702616     0.253671
Соколова Анастасия Викторовна     0.579092     0.058996
   Федорова Марина Васильевна     0.000000     0.000000
                      без А/М     0.000000     0.000000
Отчёт сохранён: C:\Users\user\Desktop\ТЗ-Data-Analysis\Prolongation_Report_2023.xlsx
